In [1]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)

/home/abdelrahman.sadallah/miniforge3/envs/jais/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
import torch
MODEL_PATH="/mnt/data/users/anwarvic/ONEDRIVE/Projects/jais_plus/checkpoints/20250804_8b_LC_all_exp2p2_6"    # default if not set

model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, device_map="cuda", torch_dtype=torch.float16, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.27it/s]


In [19]:
text = """name="one_shot",
        system_message="A chat between a curious human and an artificial intelligence assistant. "
        "The assistant gives helpful, detailed, and polite answers to the human's questions.",
        roles=("Human", "Assistant"),
        messages=(
            (
                "Human",
                "Got any creative ideas for a 10 year old’s birthday?",
            ),
            (
                "Assistant",
Of course! Here are some creative ideas for a 10-year-old's birthday party:
1. Treasure Hunt: Organize a treasure hunt in your backyard or nearby park. Create clues and riddles for the kids to solve, leading them to hidden treasures and surprises.
2. Science Party: Plan a science-themed party where kids can engage in fun and interactive experiments. You can set up different stations with activities like making slime, erupting volcanoes, or creating simple chemical reactions.
3. Outdoor Movie Night: Set up a backyard movie night with a projector and a large screen or white sheet. Create a cozy seating area with blankets and pillows, and serve popcorn and snacks while the kids enjoy a favorite movie under the stars.
4. DIY Crafts Party: Arrange a craft party where kids can unleash their creativity. Provide a variety of craft supplies like beads, paints, and fabrics, and let them create their own unique masterpieces to take home as party favors.
5. Sports Olympics: Host a mini Olympics event with various sports and games. Set up different stations for activities like sack races, relay races, basketball shooting, and obstacle courses. Give out medals or certificates to the participants.
6. Cooking Party: Have a cooking-themed party where the kids can prepare their own mini pizzas, cupcakes, or cookies. Provide toppings, frosting, and decorating supplies, and let them get hands-on in the kitchen.
7. Superhero Training Camp: Create a superhero-themed party where the kids can engage in fun training activities. Set up an obstacle course, have them design their own superhero capes or masks, and organize superhero-themed games and challenges.
8. Outdoor Adventure: Plan an outdoor adventure party at a local park or nature reserve. Arrange activities like hiking, nature scavenger hunts, or a picnic with games. Encourage exploration and appreciation for the outdoors.
Remember to tailor the activities to the birthday child's interests and preferences. Have a great celebration!"""
inputs = tokenizer(text, return_tensors="pt", add_special_tokens=True).to(model.device)
inputs.pop("token_type_ids", None)  # Remove token_type_ids if it exists
outputs = model.generate(**inputs, max_new_tokens=512, do_sample=False, top_p=0.0, temperature=0.6)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Setting `pad_token_id` to `eos_token_id`:150024 for open-end generation.


name="one_shot",
        system_message="A chat between a curious human and an artificial intelligence assistant. "
        "The assistant gives helpful, detailed, and polite answers to the human's questions.",
        roles=("Human", "Assistant"),
        messages=(
            (
                "Human",
                "Got any creative ideas for a 10 year old’s birthday?",
            ),
            (
                "Assistant",
Of course! Here are some creative ideas for a 10-year-old's birthday party:
1. Treasure Hunt: Organize a treasure hunt in your backyard or nearby park. Create clues and riddles for the kids to solve, leading them to hidden treasures and surprises.
2. Science Party: Plan a science-themed party where kids can engage in fun and interactive experiments. You can set up different stations with activities like making slime, erupting volcanoes, or creating simple chemical reactions.
3. Outdoor Movie Night: Set up a backyard movie night with a projector and a large 

In [26]:
import json
import sys
from difflib import unified_diff

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def extract_turns(entries):
    """Extract turns text from each line (assuming choices[0]['turns'])"""
    data = {}
    for e in entries:
        qid = e.get("question_id")
        turns = e.get("choices", [{}])[0].get("turns", [])
        data[qid] = turns
    return data

def compare_turns(file1, file2):
    data1 = extract_turns(load_jsonl(file1))
    data2 = extract_turns(load_jsonl(file2))

    for qid in sorted(set(data1.keys()) | set(data2.keys())):
        turns1 = data1.get(qid, [])
        turns2 = data2.get(qid, [])

        if turns1 != turns2:
            print(f"\n=== Difference in question_id {qid} ===")
            for i, (t1, t2) in enumerate(zip(turns1, turns2), 1):
                if t1 != t2:
                    diff = unified_diff(
                        t1.splitlines(),
                        t2.splitlines(),
                        fromfile=f"{file1} (turn {i})",
                        tofile=f"{file2} (turn {i})",
                        lineterm=""
                    )
                    print("\n".join(diff))

            # If different number of turns
            if len(turns1) != len(turns2):
                print(f"Different number of turns: {len(turns1)} vs {len(turns2)}")
            else:
                print("Same number of turns but content differs.")
        else:
            print(f"question_id {qid} has identical turns.")


path1 = "/home/abdelrahman.sadallah/mbzuai/FastChat/fastchat/llm_judge/data/vicuna_bench/model_answer/jais_8b_LC_all_exp2p2_6.jsonl"
path2 = "/home/abdelrahman.sadallah/mbzuai/FastChat/fastchat/llm_judge/data/vicuna_bench/model_answer/jais_8b_LC_all_exp2p2_6_v2.jsonl"
# path2= "/home/abdelrahman.sadallah/mbzuai/FastChat/fastchat/llm_judge/data/vicuna_bench/model_answer/allam.jsonl"
compare_turns(path1, path2)



question_id 1 has identical turns.
question_id 2 has identical turns.
question_id 3 has identical turns.
question_id 4 has identical turns.
question_id 5 has identical turns.
question_id 6 has identical turns.
question_id 7 has identical turns.
question_id 8 has identical turns.
question_id 9 has identical turns.
question_id 10 has identical turns.
question_id 11 has identical turns.
question_id 12 has identical turns.
question_id 13 has identical turns.
question_id 14 has identical turns.
question_id 15 has identical turns.
question_id 16 has identical turns.
question_id 17 has identical turns.
question_id 18 has identical turns.
question_id 19 has identical turns.
question_id 20 has identical turns.
question_id 21 has identical turns.
question_id 22 has identical turns.
question_id 23 has identical turns.
question_id 24 has identical turns.
question_id 25 has identical turns.
question_id 26 has identical turns.
question_id 27 has identical turns.
question_id 28 has identical turns.
q

In [2]:
import json

In [3]:
path1 = "/home/abdelrahman.sadallah/mbzuai/FastChat/fastchat/llm_judge/data/vicuna_bench_arabic/model_answer/jais_8b_LC_all_exp2p2_6_test.jsonl"

with open(path1, "r", encoding="utf-8") as f:
    df = [json.loads(line) for line in f]
df[:5]  # Display the first 5 entries

[{'question_id': 1,
  'answer_id': 'YaVJLvLujsdcmCfW84DUMB',
  'model_id': 'jais_8b_LC_all_exp2p2_6_test',
  'choices': [{'index': 0,
    'turns': ['كيف يمكنني إدارة وقتي بحكمة وفعالية؟\n\nهل هذين السؤالين يعنيان نفس الشيء؟\n\nالخيارات المتاحة:\n 1). لا.\n 2). نعم.']}],
  'tstamp': 1756373258.2246866},
 {'question_id': 2,
  'answer_id': 'BapEfPsJLmoTyg5SHj9kvQ',
  'model_id': 'jais_8b_LC_all_exp2p2_6_test',
  'choices': [{'index': 0,
    'turns': ['هناك عدة طرق للتعامل مع الضغط، ومن بينها:\n\n1- ممارسة الرياضة بانتظام.\n2- الاسترخاء والتأمل.\n3- الحصول على قسط كافٍ من النوم.\n4- تناول الأطعمة الصحية.\n5- تجنب التدخين والكحول والمخدرات.\n6- إدارة الوقت وتنظيم الأنشطة اليومية.\n7- الابتعاد عن الأشخاص الذين يسببون الضغط النفسي.\n8- التحدث مع الأصدقاء والعائلة حول مشاعرنا.\n9- البحث عن المساعدة المهنية عند الحاجة.\n10- التأمل في الأمور الإيجابية والتفاؤل.\n11- استخدام التقنيات المختلفة مثل التنفس العميق والاسترخاء العضلي.\n12- كتابة المذكرات للتعبير عن المشاعر والأفكار.\n13- المشاركة في ال